# Marmousi2 Acoustic bv1.2 Forward Modeling Validation

This notebook validates the forward-modeling side of the Marmousi2 acoustic case. Parameters, model, survey, source wavelet, and forward execution are defined directly in the notebook. No CLI runner is used.

## 1. Paths And Imports

In [ ]:
from __future__ import annotations

import csv
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ADFWI").exists():
    REPO_ROOT = Path("/liufeng1afs/project/04_Inversion/ADFWI-github")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import ADFWI
from ADFWI.model import AcousticModel
from ADFWI.propagator import AcousticPropagator, GradProcessor
from ADFWI.survey import Receiver, SeismicData, Source, Survey
from ADFWI.utils import wavelet

CASE_DIR = REPO_ROOT / "examples" / "acoustic" / "01-model-test" / "01-Marmousi2"
VALIDATION_DIR = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12"
OUTPUT_ROOT = VALIDATION_DIR / "outputs"

CASE_DIR


## 2. Local Case Definitions

These helpers are intentionally defined inside the notebook, matching the style of the original Marmousi2 example.

In [ ]:
def load_npz(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"required Marmousi2 file does not exist: {path}")
    return np.load(path, allow_pickle=True)


def optional_bound(value):
    arr = np.asarray(value)
    if arr.dtype == object:
        values = arr.tolist()
        if values is None or any(item is None for item in values):
            return None
    if arr.size != 2:
        return None
    return float(arr[0]), float(arr[1])


def cumulative_trapezoid(values: np.ndarray, dt: float) -> np.ndarray:
    out = np.zeros_like(values, dtype=np.float32)
    if values.size > 1:
        out[1:] = np.cumsum((values[:-1] + values[1:]) * (0.5 * dt), dtype=np.float64).astype(np.float32)
    return out


def build_source_wavelet(nt: int, dt: float, f0: float) -> np.ndarray:
    _, src_v = wavelet(nt, dt, f0, amp0=1)
    return cumulative_trapezoid(src_v.astype(np.float32), dt)


def build_survey(obs_npz, f0: float, *, shot_count: int | None = None, nt_samples: int | None = None) -> Survey:
    full_nt = int(obs_npz["nt"])
    nt = full_nt if nt_samples is None else int(nt_samples)
    dt = float(obs_npz["dt"])
    src_loc = np.asarray(obs_npz["src_loc"], dtype=np.int64)
    src_type = np.asarray(obs_npz["src_type"]).astype(str)
    if shot_count is not None:
        src_loc = src_loc[:shot_count]
        src_type = src_type[:shot_count]
    rcv_loc = np.asarray(obs_npz["rcv_loc"], dtype=np.int64)
    rcv_type = np.asarray(obs_npz["rcv_type"]).astype(str)

    src_v = build_source_wavelet(nt, dt, f0)
    source = Source(nt=nt, dt=dt, f0=f0)
    for (src_x, src_z), src_kind in zip(src_loc, src_type):
        source.add_source(int(src_x), int(src_z), src_v, src_type=str(src_kind))

    receiver = Receiver(nt=nt, dt=dt)
    for (rcv_x, rcv_z), rcv_kind in zip(rcv_loc, rcv_type):
        receiver.add_receiver(int(rcv_x), int(rcv_z), rcv_type=str(rcv_kind))
    return Survey(source, receiver)


def build_acoustic_model(model_npz, *, vp_grad: bool, auto_update_rho: bool, abc_type: str = "PML") -> AcousticModel:
    return AcousticModel(
        float(model_npz["ox"]),
        float(model_npz["oz"]),
        int(model_npz["nx"]),
        int(model_npz["nz"]),
        float(model_npz["dx"]),
        float(model_npz["dz"]),
        np.asarray(model_npz["vp"], dtype=np.float32),
        np.asarray(model_npz["rho"], dtype=np.float32),
        vp_bound=optional_bound(model_npz["vp_bound"]),
        rho_bound=optional_bound(model_npz["rho_bound"]),
        vp_grad=vp_grad,
        rho_grad=False,
        auto_update_rho=auto_update_rho,
        free_surface=bool(model_npz["free_surface"]),
        abc_type=abc_type,
        abc_jerjan_alpha=0.007,
        nabc=int(model_npz["nabc"]),
    )


def tensor_summary(value: torch.Tensor) -> dict[str, object]:
    detached = value.detach()
    return {
        "shape": list(detached.shape),
        "device": str(detached.device),
        "dtype": str(detached.dtype).replace("torch.", ""),
        "finite": bool(torch.isfinite(detached).all().item()),
        "min": float(detached.min().cpu().item()),
        "max": float(detached.max().cpu().item()),
        "norm": float(torch.linalg.norm(detached.reshape(-1)).cpu().item()),
    }


## 3. Parameter Definitions

In [ ]:
FORWARD_CONFIG = {
    "device": "npu:0",
    "dtype": "float32",
    "fallback_cpu": False,
    "model_file": "true_model.npz",
    "f0": 5.0,
    "forward_shot_index": 0,
    "checkpoint_segments": 10,
    "abc_type": "PML",
}
FORWARD_CONFIG


## 4. Backend Setup

In [ ]:
backend = ADFWI.set_backend(
    FORWARD_CONFIG["device"],
    dtype=FORWARD_CONFIG["dtype"],
    fallback=FORWARD_CONFIG["fallback_cpu"],
    prefer=("npu", "cpu"),
)
ADFWI.backend_diagnostics()


## 5. Model Definition

In [ ]:
model_npz = load_npz(CASE_DIR / "data" / "model" / FORWARD_CONFIG["model_file"])
model = build_acoustic_model(
    model_npz,
    vp_grad=False,
    auto_update_rho=False,
    abc_type=FORWARD_CONFIG["abc_type"],
)
model_summary = {
    "nx": model.nx,
    "nz": model.nz,
    "dx": model.dx,
    "dz": model.dz,
    "nabc": model.nabc,
    "vp": tensor_summary(model.vp),
    "rho": tensor_summary(model.rho),
}
model_summary


## 6. Observation System Definition

In [ ]:
obs_npz = load_npz(CASE_DIR / "data" / "waveform" / "obs_data.npz")
survey = build_survey(obs_npz, f0=FORWARD_CONFIG["f0"])
survey_summary = {
    "shots": survey.source.num,
    "receivers": survey.receiver.num,
    "nt": survey.source.nt,
    "dt": survey.source.dt,
    "src_x_range": [int(np.min(survey.source.get_loc()[:, 0])), int(np.max(survey.source.get_loc()[:, 0]))],
    "rcv_x_range": [int(np.min(survey.receiver.get_loc()[:, 0])), int(np.max(survey.receiver.get_loc()[:, 0]))],
}
survey_summary


## 7. Wavelet Definition

In [ ]:
source_wavelet = build_source_wavelet(survey.source.nt, survey.source.dt, FORWARD_CONFIG["f0"])
wavelet_summary = {
    "f0": FORWARD_CONFIG["f0"],
    "nt": int(source_wavelet.shape[0]),
    "dt": survey.source.dt,
    "min": float(source_wavelet.min()),
    "max": float(source_wavelet.max()),
    "norm": float(np.linalg.norm(source_wavelet)),
}
wavelet_summary


## 8. Propagator Definition

In [ ]:
propagator = AcousticPropagator(model, survey)
propagator_summary = {
    "device": str(propagator.device),
    "dtype": str(propagator.dtype).replace("torch.", ""),
    "nt": propagator.nt,
    "src_n": propagator.src_n,
    "rcv_n": propagator.rcv_n,
    "damp": tensor_summary(propagator.damp),
}
propagator_summary


## 9. Run Forward Modeling

Set `RUN_FORWARD_MODELING = True` to run the selected shot. The default is safe for inspecting the notebook setup without launching propagation.

In [ ]:
RUN_FORWARD_MODELING = False

if RUN_FORWARD_MODELING:
    if backend.name in {"cuda", "npu"}:
        backend.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        forward_record = propagator.forward(
            shot_index=np.array([FORWARD_CONFIG["forward_shot_index"]], dtype=np.int64),
            checkpoint_segments=FORWARD_CONFIG["checkpoint_segments"],
        )
    if backend.name in {"cuda", "npu"}:
        backend.synchronize()
    forward_seconds = time.perf_counter() - start
    pressure = forward_record["p"].detach()
    forward_summary = {
        "shot_index": FORWARD_CONFIG["forward_shot_index"],
        "seconds": forward_seconds,
        "pressure": tensor_summary(pressure),
    }
else:
    forward_record = None
    forward_summary = {"status": "skipped", "reason": "set RUN_FORWARD_MODELING=True"}

forward_summary
